## Define parameters

# User Configuration

In [ ]:
# Local paths -- set these before running
results_indir = ""
fmri_dir = ""
hands_annotations_dir = ""
cvat_annotations_dir = ""
mni_template_path = ""

In [ ]:
subjects = ['sub01', 'sub02','sub03','sub04','sub05','sub06']
template_subj = 'MNI152_2009_template_SSW'

## Define viz parameters

In [ ]:
import matplotlib as mpl
import json

mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Arial'
mpl.rcParams['mathtext.bf'] = 'Arial:bold'


# Map feature space names to readable labels
clean_featnames_dictmap = {
    'hand_features': 'Hand Synergy Weights',
    'action_features': 'Actions',
    'target_features': 'Target Objects',
    'interaction_features': 'Targets x Actions',
    'tool_features': 'Tools',
    'object_features': 'Passive Objects',
    'motion_features': 'Motion Energy'
}

# Define color mapping for each feature space
dict_feature_color_mapping = {
    'object_features': "#ffb41e",      # Yellow/orange
    'target_features': "#ed0d0d",      # Red
    'action_features': "#0ded1f",      # Green
    'tool_features': "#1bccf0",        # Cyan
    'interaction_features': '#8c564b', # Brown
    'hand_features': "#963ee9",        # Purple
    'motion_features': "#ee65e7"       # Pink
}


# quickflat params
quickflat_defaults = dict(roi_list=['Cortices'],
                          linecolor=(0.25,0.25,0.25),linewidth=2, labelsize=0,
                          with_colorbar=False
                          )


top_n = 10 # how many ROIs to plot in the barplor of mean per ROI


# mosaic plots
def cleanup_cbar(mosaic_plot):
    # ------- Only keep min/max values of cbar -----

    cbar = mosaic_plot._cbar

    # Get all tick labels
    ticklabels = cbar.ax.get_yticklabels()

    # Hide all but first and last
    for lab in ticklabels[1:-1]:
        lab.set_visible(False)


def custom_crosshair(nilearn_plot_img):
    '''
    Note: need to set draw_cross to False when calling plot_img for this to work
    '''
    nilearn_plot_img.draw_cross(color='black', alpha=0.5, linewidth=1)


# custom BWR map
import numpy as np
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap

def punchy_diverging(base='bwr', power=0.5, n=256):
    """
    Remap a diverging cmap so colors saturate faster away from zero.
    power < 1  -> more punch (e.g. 0.5, 0.4)
    power == 1 -> identical to base
    """
    x = np.linspace(0, 1, n)
    centered = 2 * x - 1                      # -1..1
    remapped = np.sign(centered) * np.abs(centered) ** power
    new_x = (remapped + 1) / 2
    return LinearSegmentedColormap.from_list(
        f'{base}_p{power}', cm.get_cmap(base)(new_x)
    )

punchy_bwr = punchy_diverging('bwr', power=0.75)


## Imports

In [ ]:
import os, sys

# Resolve utility paths relative to this notebook's directory
_nb_dir = os.path.abspath('')
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'speechmodeltutorial'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'utils_natcook'))

import os, itertools, sys, pickle, ast
from os.path import join
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from nilearn import image
from nilearn.masking import unmask
from nilearn.plotting import plot_img, plot_stat_map
from nilearn.image import math_img


import cortex









# ======================================= Custom imports =============================================

# ----- import local functions
# -- Speech model tutorial functions

# -- Local fmri-general functions
from get_topN_cluster_coordinates import get_topN_cluster_coordinates

# -- Natcook-specific helpers
from FMRIPathConfig import FMRIPathConfig
from banded_ridge_reconstruct_fir_weights import banded_ridge_reconstruct_fir_weights
from analyses.nilearn_utils import replace_ns_with_nan, replace_zeros_with_nan, average_list_of_imgs, nan_argmax
from analyses.mask_operations import compute_group_mask_nMinSubjects, compute_group_mask
from analyses.group_statistics import group_level_1sample_ttest, calculate_contrast_coeffs
from analyses.pycortex_utils import (
    pycortex_plot_img,
    pycortex_plot_flatmap,
    pycortex_plot_flatmap_from_vol,
    pycortex_make_volume_rgba,
    pycortex_make_vol2D_rgba,
    pycortex_make_categorical_rgba,
)


# ======================================= Input directories =============================================

# --- Define directories and paths
path_config = FMRIPathConfig(fmri_dir)
display('Path templates:', path_config.patterns)

# ======================================= Fixed parameters =============================================
# --- Other
n_jobs = -1
np.random.seed(42)


### Helper functions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

def make_pycortex_winnertakesall_legend(feature_space_names, colors, ncol, filename=None, clean_featnames_dictmap=clean_featnames_dictmap):
    """
    Create a Matplotlib legend matching the discrete colors used in pycortex.

    Parameters
    ----------
    feature_space_names : list of str
        Names of each feature space (must match indices in WTA map).
    colors : list of str
        List of hex color codes (one per feature space).
    filename : str or None
        If provided, saves the legend as an image.
    """

    # Rename feature_space_names according to dictionary
    display_names = [clean_featnames_dictmap.get(name, name) for name in feature_space_names]

    # --- Create legend figure ---
    fig, ax = plt.subplots(figsize=(2.5, len(feature_space_names) * 0.4))
    ax.axis("off")

    handles = [
        mpl.patches.Patch(color=colors[i], label=display_names[i])
        for i in range(len(feature_space_names))
    ]

    ax.legend(handles=handles,
              title="Feature Spaces",
              loc="center left",
              bbox_to_anchor=(0, 0.5),
              frameon=False,
              ncol=ncol,
              title_fontproperties={'weight': 'bold'})

    plt.tight_layout()

    if filename:
        fig.savefig(filename, dpi=300, bbox_inches="tight", transparent=True)

    return fig, ax



def drop_subcortical_rois(sorted_labels, sorted_means, sorted_stds, subcortical_rois=['Declive']):
    filtered_labels = []
    filtered_means = []
    filtered_stds = []
    dropped_labels = []

    for label, mean, std in zip(sorted_labels, sorted_means, sorted_stds):
        # case-insensitive substring match
        if any(sub_roi.lower() in label.lower() for sub_roi in subcortical_rois):
            dropped_labels.append(label)
            continue

        filtered_labels.append(label)
        filtered_means.append(mean)
        filtered_stds.append(std)

    print(f'Dropped subcortical ROIs: {dropped_labels}')

    return filtered_labels, filtered_means, filtered_stds

# Read data

### Create group results dictionary

**Read a shared brainmask_img and anat_img**

In [ ]:
# --- Check that all masks are identical. then use a random subject's mask as the group mask
all_masks = []  # list of mask images used for each subject
for subj in subjects:
    subj_mask = image.load_img(path_config.get_brainmask_path(subj))
    all_masks.append(subj_mask)


mask_shapes = [mask.shape for mask in all_masks]
assert all(np.array_equal(mask_shapes[0], m) for m in mask_shapes), "Masks shapes differ!"
mask_affines = [mask.affine for mask in all_masks]
assert all(np.array_equal(mask_affines[0], m) for m in mask_affines), "Masks affines differ!"


# --- Use a subject's mask and their anatomy as the group template
if template_subj != 'MNI152_2009_template_SSW':
    brainmask_img = image.load_img(path_config.get_brainmask_path(template_subj))
    anat_img = image.load_img(path_config.get_anat_path(template_subj))
else:
    print('reading MNI152_2009_template_SSW tempalte')
    anat_img = image.load_img(mni_template_path)
    brainmask_img = compute_group_mask(all_masks, mode='union')




In [ ]:
group_data = {'subjects':[], # to keep track which data belongs to which subj
              'explainable_variance_imgs':[], # explainable variance images
                 'uncorrected_scores_imgs':[], # R2 scores images
                 'mask_modeled_voxels_imgs':[],
                 'split_scores_imgs':{}, # one key per feature space, each a list of imgs (one img per subj)
                 'coeffs': [] # coefficints, list of dict, one per subj. keys=feature names, values=corresponding coeffs IMG
                 }


for subj in tqdm(subjects):
    # Read ridge results
    ridge_results_path =  join(results_indir, subj,  f'{subj}_ridge_results.p')
    with open(ridge_results_path, 'rb') as f:
        banded_ridge_results = pickle.load(f)

    # Save subject ID and explainable variance img for later use
    group_data['subjects'].append(subj)
    group_data['explainable_variance_imgs'].append(banded_ridge_results['explainable_variance_img'])


    # ------------------------ mask needed to reconstruct volumetric data ---------------
    subj_brainmask_img = image.load_img(path_config.get_brainmask_path(subj))
    
    # -------------------------------- R2 scores ----------------------------------------
    scores = banded_ridge_results['scores']
    scores_img = unmask(scores, subj_brainmask_img)

    mask_modelled_voxels_img = unmask(banded_ridge_results['mask_modelled_voxels'], subj_brainmask_img) # reconstruct the 3d mask for use later, and for viz here
    scores_img = replace_ns_with_nan(scores_img, mask_modelled_voxels_img) # replace non modelled voxels with nan

    group_data['uncorrected_scores_imgs'].append(scores_img)
    group_data['mask_modeled_voxels_imgs'].append(mask_modelled_voxels_img)

    # --------------------------- Split scores ---------------------------------------------
    for ii, choose_feature_space in enumerate(banded_ridge_results['feature_space_names']):
        
        # Select the feature group
        feature_space_scores = banded_ridge_results['split_scores'][ii]

        # Convert back to volume
        feature_space_scores_img = unmask(feature_space_scores, subj_brainmask_img)

        # Replace non-modelled voxels with NaN
        feature_space_scores_img = replace_ns_with_nan(feature_space_scores_img, mask_modelled_voxels_img)

        # append to group list
        if choose_feature_space not in group_data['split_scores_imgs'].keys():
            group_data['split_scores_imgs'][choose_feature_space] = []

        group_data['split_scores_imgs'][choose_feature_space].append(feature_space_scores_img)

    
    # --------------------------- Coefficients ---------------------------------------------
    # get the coefs
    subj_coeffs = banded_ridge_reconstruct_fir_weights(banded_ridge_results, verbose=False)
    # convert into a dict where the keys are the features names and the values are the corresponding coeff_img (masked  for significance)
    feats_legend = banded_ridge_results['feats_legend']

    # convert back to img
    subj_coeffs_img = unmask(subj_coeffs, mask_modelled_voxels_img)

    # Replace non modelled voxels with NaN
    subj_coeffs_img = replace_ns_with_nan(subj_coeffs_img, mask_modelled_voxels_img)
    
    subj_coeffs_dict = {feats_legend[feat_idx] : subj_coeffs_img.slicer[:,:,:,feat_idx] for feat_idx in range(len(feats_legend))}
    
    
    # append to group list
    group_data['coeffs'].append(subj_coeffs_dict)

    del banded_ridge_results




# Omnibus

### Plot average R^2 map

In [ ]:
average_scores_img = average_list_of_imgs(group_data['uncorrected_scores_imgs']) # average over subjects
average_scores_img = replace_zeros_with_nan(average_scores_img) # for plotting : replace 0s with NaN so that the 0 values are transparent


### Group stats: check for R2>0 voxelwise

In [ ]:
from scipy import stats
from nilearn.mass_univariate import permuted_ols
from nilearn.image import math_img, new_img_like
import numpy as np

# ================================================================================================
# STEP 1: Create a group mask (only voxels modeled in N subjects)
# ================================================================================================
n_min_subjects = 3

group_mask_img = compute_group_mask_nMinSubjects(group_data['mask_modeled_voxels_imgs'], n_min_subjects)


### Group stats:

In [ ]:
fwhm = 6
if not os.path.exists('group_res.p'):

    group_res = group_level_1sample_ttest(
                group_data['uncorrected_scores_imgs'],
                popmean=0,
                mask_img=group_mask_img,
                fwhm=fwhm,
                alpha=0.05,
                height_control='fdr',
                cluster_threshold=10,
                two_sided=False,
                verbose=True
                )

    with open('group_res.p', 'wb') as f:
        pickle.dump(group_res, f)
else:
    with open('group_res.p', 'rb') as f:
        group_res = pickle.load(f)

In [ ]:
# Plot the R2 average map, masked for group significance
sig_average_scores_img = replace_ns_with_nan(average_scores_img, group_res['significance_mask'])

In [ ]:
# ----------------------- flat map ------------------------------
# Plot the t-map
sig_t_map = replace_ns_with_nan(group_res['thresholded_t_map'], group_res['significance_mask'])

vmin_sig_t_map = group_res['tthreshold']
vmax = round(np.nanpercentile(sig_t_map.get_fdata(),99))


# surface
fig, ax = pycortex_plot_flatmap(sig_t_map, template_subj, 
                                cortex_Volume_kwargs=dict(vmin=vmin_sig_t_map, cmap='YlOrRd', vmax=vmax), 
                                quickflat_kwargs=quickflat_defaults,
                                cbar_label= 't-statistic')
                                # title='Significant voxels (group-level, p<.05)')




# ----------------------- mosaic plot ------------------------------
fig1, ax1 = plt.subplots(figsize=(8,4)) 
ax1.set_axis_off() # remove all ax1 borders/axis

mosaic_plot = plot_stat_map(
    sig_t_map,
    bg_img=anat_img,
    cmap='YlOrRd',
    symmetric_cbar=False,
    threshold=0,
    vmin=vmin_sig_t_map,
    vmax=vmax,
    display_mode='mosaic',
    cut_coords=4,
    black_bg=False,
    figure=fig1
)

# Add a colorbar label
mosaic_plot._cbar.set_label('t-statistic', fontsize=11, fontweight='bold')

# only keep min/max values on cbar
cleanup_cbar(mosaic_plot)




# Feature spaces

### Winner takes all map: descriptive, masked by group significance

In [ ]:
# Custom reordering of the feature spaces: affects the color mapping and the order in the legend
ordered_feature_spaces = [
    'object_features',
    'target_features', 
    'action_features', 
    'hand_features', 
    'motion_features'
    ]

# Extract colors in the order of ordered_feature_spaces
colors = [dict_feature_color_mapping[fs] for fs in ordered_feature_spaces]


In [ ]:
from nilearn.image import smooth_img

# create a list of all split scores, where each split score is the average across subjects
split_scores_averages = []
split_scores_averages_imgs = [] # can be useful later 
feature_space_names = []

for feature_space_name in ordered_feature_spaces:
    average_split_scores_img =  average_list_of_imgs(group_data['split_scores_imgs'][feature_space_name])

    # Mask out non-significant voxels using group significance mask
    average_split_scores_img = replace_ns_with_nan(average_split_scores_img, group_res['significance_mask'])

    average_split_scores_data = average_split_scores_img.get_fdata()

    # append to output lists
    split_scores_averages.append(average_split_scores_data)
    feature_space_names.append(feature_space_name) # so we keep track of indices
    split_scores_averages_imgs.append(average_split_scores_img)

split_scores_averages = np.array(split_scores_averages)
print('split_scores_averages.shape =', split_scores_averages.shape)


# get the index of the winner-takes-all at each voxel. Uses custom function, else nans are converted to 0s (which here are meaningful)
wta = nan_argmax(split_scores_averages, axis=0)  # shape (n_voxels,) a label per voxel (e.g., 0 = tools, 1 = objects, …). 

# Convert to a nilearn image
wta_img = new_img_like(brainmask_img, wta)


In [ ]:
# Create a discrete colormap from your colors
cmap_discrete = mpl.colors.ListedColormap(colors)


# --------------- plot ---------------
# Now uses discrete colormap
legend_str = "\n".join(f"{idx} {feat_name}" for idx, feat_name in enumerate(feature_space_names))

kwargs = {'vmin':0, 
          'vmax':len(feature_space_names)-1,  
          'cmap':cmap_discrete,
          'description':legend_str}


# ------ flatmap ------ 
vol = pycortex_make_categorical_rgba(
    wta_img,
    template_subj,
    colors=colors,                               # one entry per category, in label order
    alpha_img=group_res['significance_mask'],    # optional: hide non-sig voxels
    # bg_label=0,                                # or: hide a specific integer label
)

wta_flat, _ = pycortex_plot_flatmap_from_vol(
    vol,
    title='',
    quickflat_kwargs=quickflat_defaults,  # categorical → no colorbar
)



fig_legend1, _ = make_pycortex_winnertakesall_legend(feature_space_names, colors=colors, ncol=3)
fig_legend2, _ = make_pycortex_winnertakesall_legend(feature_space_names, colors=colors, ncol=5)

# ------ Mosaic plot ------ 
data = wta_img.get_fdata()

# Need to do some edits so that the model with index=0 is not automatically made transparent
# Shift all valid indices up by 1, so model 0 → 1, model 1 → 2, etc.
# This keeps NaN as NaN
data_shifted = np.where(np.isnan(data), np.nan, data + 1)
wta_shifted = new_img_like(wta_img, data_shifted)

fig1, ax1 = plt.subplots(figsize=(8,4)) 
ax1.set_axis_off() # remove all ax1 borders/axis

mosaic_plot = plot_stat_map(
    wta_shifted,
    bg_img=anat_img,
    cmap=cmap_discrete,
    colorbar=False,
    display_mode='mosaic',
    cut_coords=4,
    vmin=1,  # Start at 1 (was model 0)
    vmax=len(feature_space_names), # end one index later (was len(feature_space_names)-1),
    black_bg=False,
    figure=fig1
)


# save:
wta_flat.savefig('wta_flat.png', dpi=300, bbox_inches='tight')
mosaic_plot.savefig('wta_mosaic.png', dpi=300)
fig_legend1.savefig('wta_legend_v1.png', dpi=300, bbox_inches='tight')
fig_legend2.savefig('wta_legend_v2.png', dpi=300, bbox_inches='tight')


# Contrasts


### Coefficients contrasts

In [ ]:
def wrapper_contrast_test(feature_spaces_to_compare, group_data=group_data, group_mask_img=group_res['significance_mask'], anat_img=anat_img, template_subj=template_subj, title=None, cmap='bwr'):
    ''' Wrapper to compute contrast of coefficients between two feature spaces across subjects,
        perform group-level t-test, and plot the significant average contrast map. 
        No masking applied - all voxels are used. Masking should be done outside the function if needed.
    Parameters
    ----------
    feature_spaces_to_compare : list of str
        List of exactly two feature space names to compare (e.g., ['object_features', 'target_features']).
    group_data : dict
        Dictionary containing group-level data, including 'coeffs' key with individual subject coefficients.
    group_mask_img : NiftiImage
        Brain mask image defining voxels to analyze.
    anat_img : NiftiImage
        Anatomical image for background in plotting.
    template_subj : str
        Template subject name for pycortex plotting.
    title : str or None
        If provided, use this title for plots instead of default.
    ''' 

    # calculate the contrast
    avg_contrast_img, group_contrast_coeffs = calculate_contrast_coeffs(group_data['coeffs'], feature_spaces_to_compare)

    # Perform the group stats: 2 tailed t-test against 0
    contrast_res = group_level_1sample_ttest(
        stat_imgs=group_contrast_coeffs,
        popmean=0,
        mask_img=group_mask_img,
        fwhm=6,
        alpha=0.05,
        height_control='fdr',
        cluster_threshold=10,
        two_sided=True,
        verbose=False)

    # plot the contrast average map, masked for significance
    sig_avg_contrast_img = replace_ns_with_nan(avg_contrast_img, contrast_res['significance_mask'])
    
    
    # --------------- plot ---------------    
    vmax = np.nanpercentile(np.abs(sig_avg_contrast_img.get_fdata()), 99)  # 99th percentile of absolute values
    vmin = -vmax  # Symmetric negative

    # -------- Surface
    flatmap, _ = pycortex_plot_flatmap(sig_avg_contrast_img, template_subj, cortex_Volume_kwargs=dict(vmin=vmin,
                                                                                        vmax=vmax,
                                                                                        cmap=cmap),
                                                                                        title=title, cbar_label='Δβ',
                                                                                        quickflat_kwargs=quickflat_defaults)
    # -------- Mosaic
    fig_mosaic, ax_mosaic = plt.subplots(figsize=(8,4)) 
    ax_mosaic.set_axis_off() # remove all ax1 borders/axis
    mosaic_plot = plot_stat_map(
                    sig_avg_contrast_img,
                    bg_img=anat_img,
                    cmap=cmap,
                    symmetric_cbar=True,
                    # threshold=0,
                    vmin=vmin,
                    vmax=vmax,
                    colorbar=True,
                    display_mode='mosaic',
                    cut_coords=4,
                    title=title,
                    black_bg=False,
                    figure=fig_mosaic
                    )    
    
    mosaic_plot._cbar.set_label('Δβ', fontsize=11, fontweight='bold')
    cleanup_cbar(mosaic_plot)



    return flatmap, fig_mosaic, sig_avg_contrast_img



In [ ]:
contrasts_outdir = 'contrasts/contrasts'
if not os.path.exists(contrasts_outdir):
    os.makedirs(contrasts_outdir, exist_ok=True)


all_contrasts_to_test = [
    ['object_features', 'target_features'],
]




clean_featnames_dictmap['hands_features'] =  'Hand Posture Primitives'


for current_contrast in all_contrasts_to_test:
    contrast_name = f'{clean_featnames_dictmap[current_contrast[0]]}-{clean_featnames_dictmap[current_contrast[1]]}'

    fig_flatmap, fig_mosaic, sig_avg_contrast_img = wrapper_contrast_test(current_contrast, title=contrast_name)
    fig_flatmap.savefig(join(contrasts_outdir, f'{contrast_name}_flat.png'), bbox_inches='tight', dpi=300 )
    fig_mosaic.savefig(join(contrasts_outdir, f'{contrast_name}_mosaic.png'), bbox_inches='tight', dpi=300)

## Split score contrast

In [ ]:
# function to calculate contrast from split scores
def calculate_split_score_contrast(featspace1, featspace2, 
                                   significance_mask=group_res['significance_mask'] ,
                                   split_scores_imgs=group_data['split_scores_imgs'],
                                    cmap=punchy_bwr):
    featspace1_scores = split_scores_imgs[featspace1]
    featspace2_scores = split_scores_imgs[featspace2]

    allsubj_contrasts = []
    for idx_subj in range(len(featspace1_scores)):
        # calculate contrast for this subject
        subj_contrast = math_img('img1-img2', img1=featspace1_scores[idx_subj], img2=featspace2_scores[idx_subj])

        # Mask NS voxels
        subj_contrast = replace_ns_with_nan(subj_contrast, significance_mask)

        allsubj_contrasts.append(subj_contrast)
    
    return allsubj_contrasts




def plot_contrast(sig_avg_contrast_img, title, cmap, 
                  anat_img=anat_img, template_subj=template_subj,
                 n_jobs=n_jobs, top_n=top_n):
    """
    Plot a significance-masked contrast image in the standard set of views.
    Expects sig_avg_contrast_img to already be masked (NS voxels -> NaN).
    
    Returns
    -------
    fig_flatmap, mosaic_plot, fig_bar, df_roi, positive_peak_display, negative_peak_display
    """
    vmax = np.nanpercentile(np.abs(sig_avg_contrast_img.get_fdata()), 99)
    vmin = -vmax

    # -------- Flatmap
    fig_flatmap, _ = pycortex_plot_flatmap(
        sig_avg_contrast_img, template_subj,
        cortex_Volume_kwargs=dict(vmin=vmin, vmax=vmax, cmap=cmap),
        title=title,
        cbar_label='ΔR²',
        quickflat_kwargs=quickflat_defaults)

    # -------- Mosaic
    fig_mosaic, ax_mosaic = plt.subplots(figsize=(8, 4))
    ax_mosaic.set_axis_off()
    mosaic_plot = plot_stat_map(
        sig_avg_contrast_img,
        bg_img=anat_img,
        cmap=cmap,
        symmetric_cbar=True,
        vmin=vmin, vmax=vmax,
        colorbar=True,
        display_mode='mosaic',
        cut_coords=4,
        title=title,
        black_bg=False,
        figure=fig_mosaic)
    mosaic_plot._cbar.set_label('ΔR²', fontsize=11, fontweight='bold')
    cleanup_cbar(mosaic_plot)



    return fig_flatmap, mosaic_plot


Calculate split score contrasts: Targets vs. Objects

In [ ]:
# Calculate the contrast for each subject (masked for significance)
group_target_object_scores_contrast = calculate_split_score_contrast('target_features', 'object_features')

# Perform the group stats: 2 tailed t-test against 0
contrast_res = group_level_1sample_ttest(
    stat_imgs=group_target_object_scores_contrast,
    popmean=0,
    mask_img=group_res['significance_mask'],
    fwhm=6,
    alpha=0.05,
    height_control='fdr',
    cluster_threshold=10,
    two_sided=True,
    verbose=False)

# get the contrast average map, masked for significance
average_target_oject_scores_contrast = average_list_of_imgs(group_target_object_scores_contrast)
sig_average_target_oject_scores_contrast = replace_ns_with_nan(average_target_oject_scores_contrast, contrast_res['significance_mask'])



In [ ]:
vmax = np.nanpercentile(np.abs(sig_average_target_oject_scores_contrast.get_fdata()), 99)
vmin = -vmax

fig_mosaic, ax_mosaic = plt.subplots(figsize=(8, 4))
ax_mosaic.set_axis_off()

mosaic_plot = plot_stat_map(
    sig_average_target_oject_scores_contrast,
    bg_img=anat_img,
    cmap=punchy_bwr,
    symmetric_cbar=True,
    vmin=-0.08, vmax=0.08,
    colorbar=True,
    display_mode='mosaic',
    cut_coords=4,
    title='',
    black_bg=False,
    figure=fig_mosaic,
)
mosaic_plot._cbar.set_label('ΔR²', fontsize=11, fontweight='bold')
cleanup_cbar(mosaic_plot)

In [ ]:
fig_flatmap, _ = pycortex_plot_flatmap(
    sig_average_target_oject_scores_contrast, template_subj,
    cortex_Volume_kwargs=dict(vmin=vmin, vmax=vmax, cmap=punchy_bwr),
    # title=title,
    cbar_label='ΔR²',
    quickflat_kwargs=quickflat_defaults)
